In [1]:
import pandas as pd


In [6]:
df = pd.read_csv("../data/processed/merged_aqi_weather_hourly.csv ")
df.head()

,location_name,sensor_id,datetime,parameter,pm25,coverage.percentComplete,coverage.percentCoverage,hour,hour_utc,temperature_2m,relative_humidity_2m,wind_speed_10m,wind_direction_10m,precipitation
0,"Anand Vihar, New Delhi - DPCC",12235610,2026-04-04 19:30:00+00:00,pm25,50.0,75.0,75.0,2026-04-05 01:00:00,2026-04-04 19:00:00+00:00,22.9,66,9.3,311,0.0
1,"Anand Vihar, New Delhi - DPCC",12235610,2026-04-04 20:30:00+00:00,pm25,75.5,100.0,100.0,2026-04-05 02:00:00,2026-04-04 20:00:00+00:00,21.8,70,4.1,347,0.0
2,"Anand Vihar, New Delhi - DPCC",12235610,2026-04-04 21:30:00+00:00,pm25,44.0,50.0,50.0,2026-04-05 03:00:00,2026-04-04 21:00:00+00:00,21.4,68,3.8,275,0.0
3,"Anand Vihar, New Delhi - DPCC",12235610,2026-04-04 22:30:00+00:00,pm25,39.8,100.0,100.0,2026-04-05 04:00:00,2026-04-04 22:00:00+00:00,20.6,69,4.8,301,0.0
4,"Anand Vihar, New Delhi - DPCC",12235610,2026-04-04 23:30:00+00:00,pm25,57.3,100.0,100.0,2026-04-05 05:00:00,2026-04-04 23:00:00+00:00,19.8,74,4.5,337,0.0


In [10]:
df.shape

(3864, 14)

In [12]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3864 entries, 0 to 3863
Data columns (total 14 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   location_name             3864 non-null   str    
 1   sensor_id                 3864 non-null   int64  
 2   datetime                  3864 non-null   str    
 3   parameter                 3864 non-null   str    
 4   pm25                      3864 non-null   float64
 5   coverage.percentComplete  3864 non-null   float64
 6   coverage.percentCoverage  3864 non-null   float64
 7   hour                      3864 non-null   str    
 8   hour_utc                  3864 non-null   str    
 9   temperature_2m            3864 non-null   float64
 10  relative_humidity_2m      3864 non-null   int64  
 11  wind_speed_10m            3864 non-null   float64
 12  wind_direction_10m        3864 non-null   int64  
 13  precipitation             3864 non-null   float64
dtypes: float64(6), int6

In [16]:
df["location_name"].value_counts()

location_name
R K Puram, Delhi - DPCC          1317
Punjabi Bagh, Delhi - DPCC       1284
Anand Vihar, New Delhi - DPCC    1263
Name: count, dtype: int64

In [17]:
df = pd.read_csv("../data/processed/merged_aqi_weather_hourly.csv")
df.shape

(3864, 14)

In [18]:
df["location_name"].value_counts()

location_name
R K Puram, Delhi - DPCC          1317
Punjabi Bagh, Delhi - DPCC       1284
Anand Vihar, New Delhi - DPCC    1263
Name: count, dtype: int64

In [19]:
df.isna().sum()

location_name               0
sensor_id                   0
datetime                    0
parameter                   0
pm25                        0
coverage.percentComplete    0
coverage.percentCoverage    0
hour                        0
hour_utc                    0
temperature_2m              0
relative_humidity_2m        0
wind_speed_10m              0
wind_direction_10m          0
precipitation               0
dtype: int64

---
### Datetime cleaning and sorting

In [20]:
df["datetime"] = pd.to_datetime(df["datetime"], utc=True)
df["hour_utc"] = pd.to_datetime(df["hour_utc"], utc=True)

In [21]:
df.groupby("location_name")["pm25"].agg(["count", "mean", "min", "max"])

,count,mean,min,max
location_name,,,,
"Anand Vihar, New Delhi - DPCC",1263,77.351995,1.0,321.0
"Punjabi Bagh, Delhi - DPCC",1284,64.234346,1.0,628.0
"R K Puram, Delhi - DPCC",1317,61.498573,6.0,270.0


In [22]:
df.groupby(df["hour_utc"].dt.hour)["pm25"].mean()

hour_utc
0     87.378916
1     96.784337
2     99.958084
3     88.579167
4     76.073512
5     65.860000
6     58.360714
7     52.692814
8     48.501829
9     44.624852
10    42.931962
11    41.804516
12    43.460329
13    46.698250
14    58.965359
15    71.390850
16    79.436335
17    78.176398
18    71.606195
19    74.140060
20    70.829091
21    69.566467
22    71.429012
23    78.839394
Name: pm25, dtype: float64

In [24]:
df = df.sort_values(["location_name", "hour_utc"]).reset_index(drop=True)

In [ ]:
# now creating time based featured to learn from daily patterns

In [25]:
df["hour_of_day"] = df["hour_utc"].dt.hour
df["day_of_week"] = df["hour_utc"].dt.dayofweek
df["month"] = df["hour_utc"].dt.month
df["is_weekend"] = df["day_of_week"].isin([5, 6]).astype(int)

In [26]:
df["pm25_lag_1"] = df.groupby("location_name")["pm25"].shift(1)
df["pm25_lag_3"] = df.groupby("location_name")["pm25"].shift(3)
df["pm25_lag_6"] = df.groupby("location_name")["pm25"].shift(6)

In [27]:
df["pm25_rolling_3"] = (
    df.groupby("location_name")["pm25"]
    .transform(lambda x: x.shift(1).rolling(window=3).mean())
)

df["pm25_rolling_6"] = (
    df.groupby("location_name")["pm25"]
    .transform(lambda x: x.shift(1).rolling(window=6).mean())
)

df["pm25_rolling_12"] = (
    df.groupby("location_name")["pm25"]
    .transform(lambda x: x.shift(1).rolling(window=12).mean())
)

In [28]:
df[
    [
        "location_name",
        "hour_utc",
        "pm25",
        "pm25_lag_1",
        "pm25_lag_3",
        "pm25_rolling_3"
    ]
].head(15)

,location_name,hour_utc,pm25,pm25_lag_1,pm25_lag_3,pm25_rolling_3
0,"Anand Vihar, New Delhi - DPCC",2026-04-04 19:00:00+00:00,50.0,NaN,NaN,NaN
1,"Anand Vihar, New Delhi - DPCC",2026-04-04 20:00:00+00:00,75.5,50.0,NaN,NaN
2,"Anand Vihar, New Delhi - DPCC",2026-04-04 21:00:00+00:00,44.0,75.5,NaN,NaN
3,"Anand Vihar, New Delhi - DPCC",2026-04-04 22:00:00+00:00,39.8,44.0,50.0,56.500000
4,"Anand Vihar, New Delhi - DPCC",2026-04-04 23:00:00+00:00,57.3,39.8,75.5,53.100000
5,"Anand Vihar, New Delhi - DPCC",2026-04-05 00:00:00+00:00,61.8,57.3,44.0,47.033333
6,"Anand Vihar, New Delhi - DPCC",2026-04-05 01:00:00+00:00,85.0,61.8,39.8,52.966667
7,"Anand Vihar, New Delhi - DPCC",2026-04-05 02:00:00+00:00,119.0,85.0,57.3,68.033333
8,"Anand Vihar, New Delhi - DPCC",2026-04-05 03:00:00+00:00,106.0,119.0,61.8,88.600000
9,"Anand Vihar, New Delhi - DPCC",2026-04-05 04:00:00+00:00,88.0,106.0,85.0,103.333333


In [29]:
# prediction target

In [30]:
df["pm25_24h_ahead"] = df.groupby("location_name")["pm25"].shift(-24)

In [33]:
model_ready_df = df.dropna(
    subset=[
        "pm25_lag_1",
        "pm25_lag_3",
        "pm25_lag_6",
        "pm25_rolling_3",
        "pm25_rolling_6",
        "pm25_rolling_12",
        "pm25_24h_ahead"
    ]
).copy()
# final df

In [34]:
model_ready_df.shape

(3756, 25)

In [35]:
model_ready_df["location_name"].value_counts()

location_name
R K Puram, Delhi - DPCC          1281
Punjabi Bagh, Delhi - DPCC       1248
Anand Vihar, New Delhi - DPCC    1227
Name: count, dtype: int64

In [36]:
model_ready_df.to_csv("../data/processed/model_ready_dataset.csv", index=False)